# Mentee-Mentor Matching Analysis
Complete implementation with custom keywords and optimization

## 1. Import Libraries

In [9]:
import pandas as pd
import re
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

import sys
print(sys.executable)

Libraries imported successfully!
/Users/maltejuuljohansen/.virtualenvs/r-spacyr/bin/python


## API Configuration (OpenAI)

In [ ]:
import os
from pathlib import Path

# Load `.env` if python-dotenv is installed. Tries (1) kernel working directory, (2) this project folder,
# because Jupyter/Cursor sometimes start the kernel with cwd != the folder where the notebook lives.
_DOTENV_INSTALLED = False
try:
    from dotenv import load_dotenv

    _DOTENV_INSTALLED = True
    _CONFLUX_ROOT = Path("/Users/maltejuuljohansen/Desktop/Conflux")
    _env_paths = [Path.cwd() / ".env", _CONFLUX_ROOT / ".env"]
    _loaded_path = None
    for _p in _env_paths:
        if _p.is_file():
            load_dotenv(_p, override=False)
            _loaded_path = _p
            break
except ImportError:
    print("Install python-dotenv so `.env` is read automatically:  pip install python-dotenv")

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if not OPENAI_API_KEY and _DOTENV_INSTALLED:
    for _p in [Path.cwd() / ".env", Path("/Users/maltejuuljohansen/Desktop/Conflux/.env")]:
        if _p.is_file():
            _first = next((ln.strip() for ln in _p.read_text(encoding="utf-8").splitlines() if ln.strip()), "")
            if _first and not _first.startswith("OPENAI_API_KEY="):
                print(
                    f"Found `.env` at {_p}, but the first non-empty line must start with OPENAI_API_KEY=\n"
                    "Example:  OPENAI_API_KEY=sk-...   (no spaces around =)"
                )
            break

if OPENAI_API_KEY:
    print("OpenAI API key loaded (environment and/or .env).")
    from openai import OpenAI

    openai_client = OpenAI(api_key=OPENAI_API_KEY)
else:
    openai_client = None
    print(
        "OPENAI_API_KEY not found. Before running OpenAI cells, do one of the following:\n"
        "  • In the terminal you use to start Jupyter:  export OPENAI_API_KEY='your-key-here'\n"
        "  • Or create `.env` in this folder with:       OPENAI_API_KEY=your-key-here\n"
        "    (requires: pip install python-dotenv)\n"
        "  • In Cursor/VS Code you can also set the variable in the integrated terminal or in Run/Debug env settings."
    )

# OpenAI scoring is enabled for both quality and final rank stages.
USE_OPENAI_QUALITY = True
USE_OPENAI_RERANK = False

OpenAI API key loaded (environment and/or .env).


## 2. Load Data

In [3]:
path = "/Users/maltejuuljohansen/Desktop/Conflux/Mentee_Signup_2025.xlsx"

matches = pd.read_excel(path, sheet_name="Matches")
mentorer = pd.read_excel(path, sheet_name="Mentors")
mentees = pd.read_excel(path, sheet_name="Mentees", header=0)

print(f"Mentees: {len(mentees)}")
print(f"Mentors: {len(mentorer)}")
print(f"Existing matches: {len(matches)}")

Mentees: 246
Mentors: 114
Existing matches: 113


## 3. Clean and Prepare Data

In [4]:
mentees_renset = (
    mentees
    .sort_values("Completion time")
    .drop_duplicates(subset=["Name"], keep="last")
    .copy()
)

mentees_renset["id"] = range(len(mentees_renset))
mentees_renset["rolle"] = "mentee"
mentees_renset["navn"] = mentees_renset["Name"]
mentees_renset["studie"] = mentees_renset["${Q-P} ${Q-BO} What is the name of your line of study?"]
mentees_renset["onske_1"] = mentees_renset["Suggestion 1"]
mentees_renset["onske_2"] = mentees_renset["Suggestion 2"]
mentees_renset["onske_3"] = mentees_renset["Suggestion 3"]
mentees_renset["tekst"] = (
    mentees_renset.iloc[:, 19].astype(str) + " " +
    mentees_renset.iloc[:, 20].astype(str)
)

mentorer_renset = mentorer.copy()
mentorer_renset["id"] = range(len(mentorer_renset))
mentorer_renset["rolle"] = "mentor"
mentorer_renset["navn"] = mentorer_renset["Title"]
if "Content" not in mentorer_renset.columns:
    raise KeyError("Kolonnen 'Content' blev ikke fundet i mentorarket.")
mentorer_renset["tekst"] = mentorer_renset["Content"].fillna("").astype(str).str.strip()

print(f"Mentees: {len(mentees_renset)}, Mentors: {len(mentorer_renset)}")

Mentees: 239, Mentors: 114


## 4. Define Keywords and Stop Words

In [ ]:
import json
import random
import time
from collections import Counter

openai_client = globals().get("openai_client")
USE_OPENAI_KEYWORDS = globals().get("USE_OPENAI_KEYWORDS", True)
OPENAI_KEYWORD_MODEL = globals().get("OPENAI_KEYWORD_MODEL", "gpt-4.1-mini")
KEYWORDS_CACHE_PATH = Path(globals().get("KEYWORDS_CACHE_PATH", "/Users/maltejuuljohansen/Desktop/Conflux/generated_keywords.json"))
FORCE_REFRESH_KEYWORDS = globals().get("FORCE_REFRESH_KEYWORDS", False)

FALLBACK_BIGRAMS = [
    "machine_learning", "product_development", "water_engineering", "life_science", "data_science",
    "data_analysis", "data_analytics", "data_scientist", "data_engineer", "wind_energy",
    "product_design", "product_management", "supply_chain", "management_consulting",
    "computer_science", "software_engineering", "software_engineer", "software_development", "data_engineering",
    "artificial_intelligence", "cloud_computing", "cyber_security", "embedded_systems",
    "database_systems", "backend_development", "back_end", "front_end",
    "frontend_development", "business_intelligence", "decision_support",
    "risk_management", "renewable_energy", "novo_nordisk",
    "electrical_engineering", "mechanical_engineering", "chemical_engineering", "civil_engineering",
    "environmental_engineering", "process_engineering", "biomedical_engineering", "medical_device",
    "drug_development", "bioinformatics", "robotics_engineering",
    "industrial_engineering", "data_platform", "data_pipeline", "data_modeling",
    "predictive_modeling", "software_architecture", "system_design",
    "digital_transformation", "generative_ai", "feature_engineering",
    "decision_making", "green_transition", "energy_transition",
    "offshore_wind", "thermal_energy", "structural_engineering", "systems_engineering",
    "health_tech", "health_care", "molecular_biology",
    "genetic_engineering", "bioprocess_engineering", "pharmaceutical_development",
    "supply_chain_management", "real_world_experience",
    "business_development", "market_analysis", "project_management", "change_management",
    "strategy_consulting", "operations_management", "corporate_finance", "investment_banking",
    "asset_management", "portfolio_management", "financial_modeling", "financial_analysis",
    "private_equity", "venture_capital", "capital_markets", "mergers_acquisitions",
]

FALLBACK_ENKELTORD = [
    "iot", "biotechnology", "ui", "programs", "programming", "ai",
    "cybersecurity", "bioinformatics", "statistics", "analytical",
    "marketing", "finance", "calculations", "pharma", "hydrogen", "mathematics",
    "telemedicine", "disease", "analysing", "physics", "nanostructure", "quantum", "logistics",
    "military", "sales", "chemistry", "modelling", "mathematical", "innovation", "transportation",
    "therapist", "ux", "app", "pharmacology", "biotechnological", "biopharmaceuticals", "robot",
    "pharmaceutical", "code", "automation", "optimization", "algorithm", "algorithms",
    "simulation", "consulting", "sustainability",
]

FALLBACK_STOPORD = {
    "i'm", "student", "dtu", "h2", "denmark", "time", "understand", "understanding", "mentee", "i've", "lot",
    "study", "studies", "studied", "students", "university", "new", "using", "paths", "fast", "person",
    "school", "class", "semester", "learn", "learning", "education", "academic", "companies",
    "1", "2", "3", "4", "5", "6", "7", "8", "9", "months", "january", "february", "march", "april", "may",
    "june", "july", "august", "september", "october", "november", "december", "mentor", "moment", "recently",
    "develop", "field", "background", "skill", "impact", "gain", "profession", "solid",
    "seek", "start", "term", "intern", "job", "experience", "before", "needed", "years", "year", "very",
    "because", "previous", "seeking", "og", "motivation", "de",
    "coupled", "chosen", "stopord", "cv", "conflux", "studying", "relevant", "related", "similar", "munich",
    "copenhagen", "daily", "combined", "sit", "current", "currently", "work", "working", "want", "worked", "danish", "like", "help",
    "way", "ways", "view", "views", "larger", "different", "general", "important", "interesting", "future", "personal", "professional",
    "passion", "excited", "opportunity", "opportunities", "journey", "growth", "growing", "improve", "improving", "explore", "exploring", "curiosity",
    "towards", "within", "across", "around", "through", "based", "focus", "focused", "approach", "approaches", "perspective", "perspectives",
    "motivated", "driven", "structured", "independent", "collaborative", "dedicated", "network", "career", "guidance",
    "trying", "try", "tries", "tried", "takes", "take", "taking", "taken", "end", "ending", "ended",
    "feel", "feels", "feeling", "felt", "really", "overall", "thing", "things", "thought", "thinking", "thinks",
    "enjoyed", "enjoy", "enjoys", "prefer", "prefers", "preferred", "quite", "ones", "one", "maybe", "hard",
    "company", "use", "uses", "used", "pursue", "pursues", "pursuing", "pursued", "make",
    "good", "better", "best", "great", "well", "many", "much", "more", "most", "less", "least",
    "another", "others", "someone", "something", "anything", "everything", "looking",
    "always", "often", "sometimes", "possible", "possibly", "probably", "user", "means",
    "rather", "instead", "however", "whether", "especially", "including", "include", "included", "includes",
    "example", "examples", "able", "value", "valuable", "systems", "particularly", "share", "finding", "limited",
}

KEYWORDS_SYSTEM_PROMPT = """You build keyword lists for mentee-mentor profile matching at a Danish university mentorship program.

Return ONLY valid JSON with exactly these keys:
- "bigrams": array of 40-90 snake_case multi-word domain terms (e.g. machine_learning, supply_chain)
- "single_words": array of 25-60 single tokens (skills, fields, tools, industries)
- "custom_stop_words": array of 80-180 lowercase tokens to EXCLUDE from overlap matching

Rules:
- Focus on academic/professional matching value: study fields, research themes, tools, methods, industries, roles.
- Exclude person names, company names used as people, months, numbers, and generic motivation boilerplate.
- Include Danish/English variants where relevant (e.g. og is a stop word).
- Bigrams must use underscores, not spaces.
- Do not wrap JSON in markdown."""


def _normalize_token_list(values):
    cleaned = []
    seen = set()
    for raw in values or []:
        token = str(raw).strip().lower().replace("-", "_").replace(" ", "_")
        token = re.sub(r"[^a-z0-9_]", "", token)
        if len(token) < 2 or token in seen:
            continue
        seen.add(token)
        cleaned.append(token)
    return cleaned


def _discover_frequent_bigrams(texts, top_n=100):
    counts = Counter()
    for text in texts:
        words = re.findall(r"\b[a-z]{2,}\b", str(text).lower())
        for i in range(len(words) - 1):
            counts[f"{words[i]}_{words[i+1]}"] += 1
    return [bg for bg, _ in counts.most_common(top_n)]


def _sample_corpus_texts(mentees_df, mentors_df, max_mentees=35, max_mentors=35, seed=42):
    rng = random.Random(seed)
    mentee_texts = mentees_df["tekst"].dropna().astype(str).tolist()
    mentor_texts = mentors_df["tekst"].dropna().astype(str).tolist()
    rng.shuffle(mentee_texts)
    rng.shuffle(mentor_texts)
    samples = mentee_texts[:max_mentees] + mentor_texts[:max_mentors]
    return [t[:1800] for t in samples if t.strip()]


def _load_keyword_cache(cache_path):
    if not cache_path.is_file():
        return None
    try:
        data = json.loads(cache_path.read_text(encoding="utf-8"))
        if not all(k in data for k in ("bigrams", "single_words", "custom_stop_words")):
            return None
        return data
    except (json.JSONDecodeError, OSError):
        return None


def _save_keyword_cache(cache_path, bigrams, single_words, custom_stop_words, meta=None):
    payload = {
        "bigrams": bigrams,
        "single_words": single_words,
        "custom_stop_words": sorted(custom_stop_words),
        "meta": meta or {},
    }
    cache_path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")


def generate_keywords_openai(mentees_df, mentors_df, cache_path, force_refresh=False):
    if not force_refresh:
        cached = _load_keyword_cache(cache_path)
        if cached:
            print(f"Loaded cached keywords from {cache_path}")
            return (
                _normalize_token_list(cached["bigrams"]),
                _normalize_token_list(cached["single_words"]),
                set(_normalize_token_list(cached["custom_stop_words"])),
            )

    if openai_client is None:
        raise RuntimeError("openai_client is not initialized. Run the API Configuration cell first.")

    all_texts = (
        mentees_df["tekst"].dropna().astype(str).tolist()
        + mentors_df["tekst"].dropna().astype(str).tolist()
    )
    sample_texts = _sample_corpus_texts(mentees_df, mentors_df)
    frequent_bigrams = _discover_frequent_bigrams(all_texts, top_n=80)

    corpus_block = "\n\n---\n\n".join(f"Sample {i+1}:\n{t}" for i, t in enumerate(sample_texts[:25]))
    user_prompt = f"""Corpus stats: {len(mentees_df)} mentees, {len(mentors_df)} mentors.

Most frequent two-word patterns in corpus (snake_case hints):
{", ".join(frequent_bigrams[:60])}

Representative profile texts:
{corpus_block}
"""

    last_error = None
    for attempt in range(1, 4):
        try:
            response = openai_client.chat.completions.create(
                model=OPENAI_KEYWORD_MODEL,
                temperature=0,
                response_format={"type": "json_object"},
                messages=[
                    {"role": "system", "content": KEYWORDS_SYSTEM_PROMPT},
                    {"role": "user", "content": user_prompt},
                ],
            )
            raw = (response.choices[0].message.content or "").strip()
            data = json.loads(raw)
            bigrams = _normalize_token_list(data.get("bigrams", []))
            single_words = _normalize_token_list(data.get("single_words", []))
            custom_stop = set(_normalize_token_list(data.get("custom_stop_words", [])))
            if not bigrams and not single_words:
                raise ValueError(f"OpenAI returned empty keyword lists: {raw[:300]}")
            _save_keyword_cache(
                cache_path,
                bigrams,
                single_words,
                custom_stop,
                meta={"model": OPENAI_KEYWORD_MODEL, "attempt": attempt},
            )
            print(f"OpenAI generated keywords and saved cache to {cache_path}")
            return bigrams, single_words, custom_stop
        except Exception as exc:
            last_error = exc
            if attempt < 3:
                time.sleep(1.5 * attempt)
            else:
                raise RuntimeError(f"OpenAI keyword generation failed after 3 attempts: {last_error}") from exc


def apply_fallback_keywords():
    return list(FALLBACK_BIGRAMS), list(FALLBACK_ENKELTORD), set(FALLBACK_STOPORD)


if USE_OPENAI_KEYWORDS and openai_client is not None:
    try:
        mine_bigrams, mine_enkeltord, mine_stopord_r = generate_keywords_openai(
            mentees_renset,
            mentorer_renset,
            KEYWORDS_CACHE_PATH,
            force_refresh=FORCE_REFRESH_KEYWORDS,
        )
        print(
            f"OpenAI keywords ready: {len(mine_bigrams)} bigrams, "
            f"{len(mine_enkeltord)} single words, {len(mine_stopord_r)} custom stop words"
        )
    except Exception as exc:
        print(f"OpenAI keyword generation failed ({exc}); using built-in fallback lists.")
        mine_bigrams, mine_enkeltord, mine_stopord_r = apply_fallback_keywords()
else:
    print("USE_OPENAI_KEYWORDS is off or openai_client missing — using built-in fallback lists.")
    mine_bigrams, mine_enkeltord, mine_stopord_r = apply_fallback_keywords()

mine_vigtige_ord = mine_bigrams + mine_enkeltord

stop_words = set(ENGLISH_STOP_WORDS)
stop_words.update(mine_stopord_r)

print("Keywords and stop words defined")


## 5. Apply Bigrams

In [6]:
def indsæt_bigrams(tekst, bigrams):
    for bg in bigrams:
        tekst = re.sub(bg.replace("_", " "), bg, tekst, flags=re.IGNORECASE)
    return tekst

mentees_renset["tekst"] = mentees_renset["tekst"].apply(lambda x: indsæt_bigrams(x, mine_bigrams))
mentorer_renset["tekst"] = mentorer_renset["tekst"].apply(lambda x: indsæt_bigrams(x, mine_bigrams))
print("Bigrams applied!")

Bigrams applied!


## 6. Generate Embeddings

OpenAI `text-embedding-3-large` (run the **API Configuration** cell first so `openai_client` is set).

In [7]:
import time
import numpy as np
from sklearn.preprocessing import normalize

openai_client = globals().get("openai_client")
if openai_client is None:
    raise RuntimeError(
        "openai_client is not initialized. Run the API Configuration cell and set OPENAI_API_KEY."
    )

OPENAI_EMBEDDING_MODEL = "text-embedding-3-large"
EMBED_BATCH_SIZE = 64


def _clean_text_for_embed(t):
    if t is None or (isinstance(t, float) and pd.isna(t)):
        return " "
    s = str(t).strip()
    return s if s else " "


def embed_texts_openai(texts, label="items"):
    vectors = []
    n = len(texts)
    for start in range(0, n, EMBED_BATCH_SIZE):
        end = min(start + EMBED_BATCH_SIZE, n)
        batch = [_clean_text_for_embed(t) for t in texts[start:end]]
        last_error = None
        for attempt in range(1, 4):
            try:
                resp = openai_client.embeddings.create(
                    model=OPENAI_EMBEDDING_MODEL,
                    input=batch,
                )
                ordered = sorted(resp.data, key=lambda d: d.index)
                vectors.extend([d.embedding for d in ordered])
                print(f"  {label}: {end}/{n}")
                break
            except Exception as exc:
                last_error = exc
                if attempt < 3:
                    time.sleep(1.5 * attempt)
                else:
                    raise RuntimeError(
                        f"OpenAI embedding request failed after 3 attempts: {last_error}"
                    ) from last_error
    arr = np.asarray(vectors, dtype=np.float32)
    return normalize(arr, norm="l2")


print(f"Encoding mentors via {OPENAI_EMBEDDING_MODEL}...")
mentor_embeddings = embed_texts_openai(mentorer_renset["tekst"].tolist(), label="mentors")

print(f"Encoding mentees via {OPENAI_EMBEDDING_MODEL}...")
mentee_embeddings = embed_texts_openai(mentees_renset["tekst"].tolist(), label="mentees")

print("Computing cosine similarity for retrieval...")
similarities = cosine_similarity(mentee_embeddings, mentor_embeddings)
print(f"Similarity matrix: {similarities.shape}")

Loading bi-encoder model...


Encoding mentors...


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Encoding mentees...


Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Computing cosine similarity for retrieval...
Similarity matrix: (239, 114)


## 7. Name Mappings and Helper Functions

In [ ]:
mentee_names = mentees_renset["navn"].tolist()
mentor_names = mentorer_renset["navn"].tolist()
mentor_name_to_index = {navn: i for i, navn in enumerate(mentor_names)}

rigtige_matches = (
    matches[["Menteee name", "Mentor name"]]
    .dropna(subset=["Menteee name", "Mentor name"])
    .drop_duplicates(subset=["Menteee name"])
    .rename(columns={"Menteee name": "mentee", "Mentor name": "mentor_rigtig"})
)

SHARED_KEYWORDS_SYSTEM = """You extract shared academic/professional profile keywords from a mentee-mentor pair.

Return ONLY a comma-separated list of 3-8 concrete overlap terms (study fields, research themes, tools, methods, industries, roles).
Use lowercase. Use underscores for multi-word terms (e.g. machine_learning, supply_chain).
Exclude generic filler (student, mentor, mentee, experience, passion, dtu, denmark, university, career, guidance).
If there is almost no meaningful overlap, return an empty string.
No explanation. No numbering."""


def find_shared_terms_local(tekst1, tekst2, top_n=5):
    tokens1 = set(re.findall(r"\b[a-z_]{3,}\b", str(tekst1).lower()))
    tokens2 = set(re.findall(r"\b[a-z_]{3,}\b", str(tekst2).lower()))
    fælles = [ord for ord in tokens1 & tokens2 if ord not in stop_words]
    fælles.sort(key=lambda x: (x in mine_vigtige_ord, x), reverse=True)
    return ", ".join(fælles[:top_n]) if fælles else ""


def find_shared_terms_openai(tekst1, tekst2, top_n=8):
    client = globals().get("openai_client")
    if client is None:
        return find_shared_terms_local(tekst1, tekst2, top_n)

    model = globals().get("OPENAI_KEYWORD_MODEL", "gpt-4.1-mini")
    user_prompt = (
        f"Mentee profile text:\n{str(tekst1)[:2500]}\n\n"
        f"Mentor profile text:\n{str(tekst2)[:2500]}"
    )
    last_error = None
    for attempt in range(1, 4):
        try:
            response = client.chat.completions.create(
                model=model,
                temperature=0,
                max_tokens=120,
                messages=[
                    {"role": "system", "content": SHARED_KEYWORDS_SYSTEM},
                    {"role": "user", "content": user_prompt},
                ],
            )
            raw = (response.choices[0].message.content or "").strip()
            terms = []
            for part in raw.split(","):
                token = part.strip().lower().replace("-", "_").replace(" ", "_")
                token = re.sub(r"[^a-z0-9_]", "", token)
                if len(token) >= 2 and token not in stop_words:
                    terms.append(token)
            return ", ".join(terms[:top_n])
        except Exception as exc:
            last_error = exc
            if attempt < 3:
                time.sleep(1.0 * attempt)
            else:
                print(f"OpenAI shared-keywords fallback ({last_error})")
                return find_shared_terms_local(tekst1, tekst2, top_n)


def find_shared_terms(tekst1, tekst2, top_n=5):
    if globals().get("USE_OPENAI_SHARED_KEYWORDS") and globals().get("openai_client") is not None:
        return find_shared_terms_openai(tekst1, tekst2, top_n)
    return find_shared_terms_local(tekst1, tekst2, top_n)

mentee_text_map = dict(zip(mentees_renset["navn"], mentees_renset["tekst"]))
mentor_text_map = dict(zip(mentorer_renset["navn"], mentorer_renset["tekst"]))

print("Setup complete")

Setup complete


## 8. Build Match Candidates

In [9]:
all_match_candidates = []
for i, mentee_name in enumerate(mentee_names):
    for mentor_idx, mentor_name in enumerate(mentor_names):
        all_match_candidates.append({
            "mentee": mentee_name,
            "mentor": mentor_name,
            "similarity_score": round(float(similarities[i, mentor_idx]), 4)
        })

all_match_candidates = pd.DataFrame(all_match_candidates)
all_match_candidates = all_match_candidates.merge(rigtige_matches, on="mentee", how="left")
all_match_candidates["er_rigtigt_match"] = all_match_candidates["mentor"] == all_match_candidates["mentor_rigtig"]

onske1_count = mentees_renset["onske_1"].value_counts()
onske2_count = mentees_renset["onske_2"].value_counts()
onske3_count = mentees_renset["onske_3"].value_counts()

# Calculate popular counts for each mentee
all_match_candidates["onske_1_popularitet"] = all_match_candidates["mentee"].apply(
    lambda x: onske1_count.get(mentees_renset[mentees_renset["navn"] == x]["onske_1"].values[0], 0) if len(mentees_renset[mentees_renset["navn"] == x]) > 0 and pd.notna(mentees_renset[mentees_renset["navn"] == x]["onske_1"].values[0]) else 0
)
all_match_candidates["onske_2_popularitet"] = all_match_candidates["mentee"].apply(
    lambda x: onske2_count.get(mentees_renset[mentees_renset["navn"] == x]["onske_2"].values[0], 0) if len(mentees_renset[mentees_renset["navn"] == x]) > 0 and pd.notna(mentees_renset[mentees_renset["navn"] == x]["onske_2"].values[0]) else 0
)
all_match_candidates["onske_3_popularitet"] = all_match_candidates["mentee"].apply(
    lambda x: onske3_count.get(mentees_renset[mentees_renset["navn"] == x]["onske_3"].values[0], 0) if len(mentees_renset[mentees_renset["navn"] == x]) > 0 and pd.notna(mentees_renset[mentees_renset["navn"] == x]["onske_3"].values[0]) else 0
)

print(f"Built {len(all_match_candidates)} mentee-mentor candidates")

Built 27246 mentee-mentor candidates


## 9. Video score (for OpenAI quality blend)

In [10]:
# Video flag and score (blended into OpenAI quality_score_final in the next section)
mentees_renset["video"] = mentees_renset.iloc[:, 21].astype(str).str.contains("https", case=False, na=False).astype(int)
mentees_renset["video_score"] = mentees_renset["video"] * 10

visible_score_cols = [
    "video_score"
]
existing_score_cols = [col for col in visible_score_cols if col in all_match_candidates.columns]
if existing_score_cols:
    all_match_candidates = all_match_candidates.drop(columns=existing_score_cols)

all_match_candidates = all_match_candidates.merge(
    mentees_renset[[
        "navn",
        "video_score"
    ]],
    left_on="mentee",
    right_on="navn",
    how="left"
).drop(columns=["navn"])

## 10. OpenAI Quality Score

In [ ]:
import re
import time

# Set in the API Configuration cell; avoids NameError if that cell was not run yet.
openai_client = globals().get("openai_client")

USE_OPENAI_QUALITY = True
OPENAI_QUALITY_MODEL = "gpt-4.1-mini"

QUALITY_SYSTEM_PROMPT = """You are a strict admissions evaluator.

Your job is NOT to be helpful or encouraging. Your job is to judge quality.

You will see three survey questions (as provided) and the student's answers. Give ONE holistic score from 0 to 10 for the combined written application (all three answers together).

Score using the criteria below:

1. RELEVANCE (0–4 points)
- Does each answer address its stated question?
- Or is it generic, off-topic, or mostly boilerplate?

2. CONCRETENESS (0–3 points)
- Does the student give specific experiences, examples, or goals?
- Or is it vague and abstract?

3. DEPTH (0–3 points)
- Does the writing show reflection, motivation, or substance?
- Or is it superficial?

IMPORTANT RULES:
- Use ONLY the questions and answers shown in the user message. Do not invent facts about the student.
- Judge substance in whatever language the student wrote in (do not penalize for Danish vs English).
- Do NOT treat longer answers as automatically better; reward specificity and substance, not word count.
- Be strict. Most combined applications should NOT score above 7.
- A score of 9–10 is extremely rare and requires excellent depth and specificity across the answers that matter.
- Penalize vague, generic, or repetitive answers heavily.
- Be objective and critical.

Return ONLY a single number from 0 to 10.
No explanation. No text."""

def _extract_score(raw_text, prefer_score_line=False):
    text = str(raw_text).strip().replace(",", ".")
    if prefer_score_line:
        m = re.search(r"(?i)score:\s*(-?\d+(?:\.\d+)?)", text)
        if m:
            score = float(m.group(1))
            return max(0.0, min(10.0, round(score, 2)))
    match = re.search(r"-?\d+(?:\.\d+)?", text)
    if not match:
        raise ValueError(f"No numeric score found in response: {raw_text}")
    score = float(match.group(0))
    return max(0.0, min(10.0, round(score, 2)))


def _openai_numeric_score(system_prompt, user_prompt, model_name, retries=3, prefer_score_line=False):
    if openai_client is None:
        raise RuntimeError(
            "openai_client is not initialized. Set OPENAI_API_KEY (see the API Configuration cell)."
        )

    last_error = None
    for attempt in range(1, retries + 1):
        try:
            response = openai_client.chat.completions.create(
                model=model_name,
                temperature=0,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt},
                ],
            )
            raw = response.choices[0].message.content
            return _extract_score(raw, prefer_score_line=prefer_score_line)
        except Exception as exc:
            last_error = exc
            if attempt < retries:
                time.sleep(1.5 * attempt)
            else:
                raise RuntimeError(f"OpenAI scoring failed after {retries} attempts: {exc}") from exc


def build_quality_prompt(row):
    cols = mentees_renset.columns
    q20 = str(cols[19]).strip()
    q21 = str(cols[20]).strip()
    q23 = str(cols[22]).strip()

    answer20 = str(row.iloc[19]) if pd.notna(row.iloc[19]) else ""
    answer21 = str(row.iloc[20]) if pd.notna(row.iloc[20]) else ""
    answer23 = str(row.iloc[22]) if pd.notna(row.iloc[22]) else ""

    return f"""Score the following application (three question-and-answer pairs). Use each question to judge relevance of the answer directly below it.

--- Question 1 ---
{q20}

Answer 1:
{answer20}

--- Question 2 ---
{q21}

Answer 2:
{answer21}

--- Question 3 ---
{q23}

Answer 3:
{answer23}""".strip()


def score_quality_openai(row):
    prompt = build_quality_prompt(row)
    return _openai_numeric_score(QUALITY_SYSTEM_PROMPT, prompt, OPENAI_QUALITY_MODEL)


if USE_OPENAI_QUALITY:
    mentees_renset["quality_score_openai"] = mentees_renset.apply(score_quality_openai, axis=1)
    # 90% OpenAI text quality, 10% video_score.
    mentees_renset["quality_score_final"] = (
        mentees_renset["quality_score_openai"] * 0.9 + mentees_renset["video_score"] * 0.1
    ).round(2)
else:
    raise RuntimeError("USE_OPENAI_QUALITY must be True for this workflow.")

visible_score_cols = [
    "quality_score_final",
    "quality_score_openai",
    "video_score"
]
existing_score_cols = [col for col in visible_score_cols if col in all_match_candidates.columns]
if existing_score_cols:
    all_match_candidates = all_match_candidates.drop(columns=existing_score_cols)

all_match_candidates = all_match_candidates.merge(
    mentees_renset[[
        "navn",
        "quality_score_final",
        "quality_score_openai",
        "video_score"
    ]],
    left_on="mentee",
    right_on="navn",
    how="left"
).drop(columns=["navn"])

print("OpenAI quality scoring complete.")

OpenAI quality template prepared (disabled).


## 11. Total Score (Pre-Rerank)

In [12]:
# Formula: Total Score = (Cosine * 50%) + (Quality * 30%) + (Preference * 20%)
# quality_score_final = 90% OpenAI text score + 10% video_score (same 90/10 split as the old ML answer vs video weights).

openai_rerank_top_k = 3
mentee_mentor_cap = 5

all_match_candidates["cosine_norm"] = (all_match_candidates["similarity_score"] * 10).round(2)
all_match_candidates["retrieval_rank"] = (
    all_match_candidates.groupby("mentee")["similarity_score"]
    .rank(method="first", ascending=False)
    .astype(int)
)

def calculate_onske_bonus(row):
    mentor = row["mentor"]
    mentee = row["mentee"]

    mentee_data = mentees_renset[mentees_renset["navn"] == mentee]
    if len(mentee_data) == 0:
        return 0.0

    onske_1 = mentee_data.iloc[0]["onske_1"]
    onske_2 = mentee_data.iloc[0]["onske_2"]
    onske_3 = mentee_data.iloc[0]["onske_3"]

    if mentor == onske_1:
        return 10.0
    if mentor == onske_2:
        return 7.0
    if mentor == onske_3:
        return 4.0
    return 0.0

scored_candidates = all_match_candidates.copy()
scored_candidates["onske_bonus"] = scored_candidates.apply(calculate_onske_bonus, axis=1)
scored_candidates["total_score"] = (
    (scored_candidates["cosine_norm"] * 0.5) +
    (scored_candidates["quality_score_final"] * 0.3) +
    (scored_candidates["onske_bonus"] * 0.2)
).round(2)

pre_rerank_candidates = (
    scored_candidates
    .sort_values(["mentee", "total_score", "cosine_norm", "similarity_score"], ascending=[True, False, False, False])
    .groupby("mentee", group_keys=False)
    .head(openai_rerank_top_k)
    .copy()
)

print("Pre-rerank scoring complete.")
print(f"Top-k candidates per mentee (per-mentee shortlist): {openai_rerank_top_k}")

Pre-rerank scoring complete.
Top-k candidates per mentee for OpenAI reranking: 10


## 12. Final ordering, optional LLM rerank, and export table

In [ ]:
import time

openai_client = globals().get("openai_client")

# LLM rerank on every pre-selected pair (expensive). Off for now: final_rank_score = total_score.
USE_OPENAI_RERANK = False
# Short natural-language rationale only for rows in the exported "Top 3 mentees per mentor" table.
USE_OPENAI_TOP_MATCH_EXPLANATION = True

OPENAI_RERANK_MODEL = "gpt-4.1-mini"
EXPLAIN_TOP_MATCH_MODEL = OPENAI_RERANK_MODEL

FINAL_RANK_SYSTEM_PROMPT = """You are a strict matching evaluator for a mentor-student program.

Your job is to evaluate how strong the match is between a student and a mentor, using ONLY the two profile texts in the user message.

Score from 0 to 10 based on:

1. TOPIC ALIGNMENT (0–5 points)
- Do the student's stated interests align with the mentor's stated expertise and experience?

2. LEARNING POTENTIAL (0–3 points)
- Would the student realistically benefit from this mentor, given what is actually written?

3. SPECIFIC FIT (0–2 points)
- Is there clear, concrete overlap in domain, methods, tools, or goals (not just generic motivation)?

IMPORTANT RULES:
- Grounding: do not invent employers, degrees, skills, or interests that are not stated or clearly implied in the texts. If a text is thin or generic, score conservatively.
- Be strict: most random or weak matches should score below 5.
- Scores above 8 should only be given if there is a very clear strong fit supported by specifics in BOTH texts.
- When fit is ambiguous, slightly favor clearly stated overlap in domain, tools, or methods over generic enthusiasm.
- Focus on match quality supported by the text, not on how impressive either side sounds in the abstract.

Also provide ONE short sentence explaining the score.

The explanation MUST:
- justify the numeric score in terms of match strength
- reference what is (or is not) supported by the two profiles
- be specific (not generic praise)

Return in this format:
Score: <number>
Reason: <one short sentence>"""


def build_final_rank_prompt(mentee_text, mentor_text):
    return f"""Below are the mentee's and mentor's own written profiles. Judge match strength using ONLY this content (do not use outside knowledge about people, companies, or fields).

Mentee profile:
{mentee_text}

Mentor profile:
{mentor_text}""".strip()


def score_pair_openai(mentee_text, mentor_text):
    prompt = build_final_rank_prompt(mentee_text, mentor_text)
    return _openai_numeric_score(
        FINAL_RANK_SYSTEM_PROMPT, prompt, OPENAI_RERANK_MODEL, prefer_score_line=True
    )


EXPLAIN_TOP_MATCH_SYSTEM = """Write exactly ONE line of prose (a single line in the file: no line breaks, no bullet lists).

Focus almost entirely on concrete academic and professional overlap: fields of study, research themes, tools/methods, industry or role type, seniority/career stage, coursework or project experience—only where the supplied facts or keyword overlap plausibly support it.

Style rules:
- Do NOT start with "The mentee", "The mentor", "The mentee and mentor", "This pairing", or "They". Begin directly with the subject matter (domains, topics, experience).
- Do NOT mention cosine, similarity scores, ranks, decimals, "/10", "0–1", "embedding", or other numeric retrieval metrics. Describe fit in plain words only (e.g. strong / moderate / weak overlap).
- Do NOT mention video, links, or wish-list / preference.
- Optional: at most a few words on whether the written applications feel thin vs detailed—only if it changes how much we can infer about academic fit.

Use only the facts given. Do not invent employers, degrees, or skills not grounded in the overlap or topic headers. Output only that single line."""


def _snippet_label(text, max_len=95):
    t = str(text).replace("\n", " ").strip()
    if len(t) <= max_len:
        return t
    return t[: max_len - 1].rstrip() + "…"


def _profile_alignment_words(row):
    """Qualitative only — avoids echoing cosine numbers in the model reply."""
    try:
        cn = float(row.get("cosine_norm"))
    except (TypeError, ValueError):
        return "unknown overall profile-text alignment"
    if cn >= 8.2:
        return "strong overall profile-text alignment"
    if cn >= 6.8:
        return "moderate overall profile-text alignment"
    return "limited overall profile-text alignment"


def _retrieval_place_words(row):
    try:
        r = int(row.get("retrieval_rank"))
    except (TypeError, ValueError):
        return "unknown how central this mentor is among all mentors for this mentee's profile text"
    if r <= 3:
        return "among this mentee's closest few mentors by profile-text match"
    if r <= 12:
        return "a mid-ranked mentor for this mentee by profile-text match"
    return "a lower-ranked mentor for this mentee by profile-text match"


def _application_depth_words(row):
    try:
        qo = float(row.get("quality_score_openai"))
    except (TypeError, ValueError):
        return "unknown depth of written application answers"
    if qo >= 7.5:
        return "written answers look fairly detailed"
    if qo >= 5.5:
        return "written answers look moderately detailed"
    return "written answers look quite thin"


def build_match_explanation_facts(row):
    cols = mentees_renset.columns
    q20 = _snippet_label(cols[19])
    q21 = _snippet_label(cols[20])
    q23 = _snippet_label(cols[22])
    mt = mentee_text_map.get(row["mentee"], "")
    mo = mentor_text_map.get(row["mentor"], "")
    overlap = find_shared_terms(mt, mo, top_n=8)
    if not str(overlap).strip():
        overlap = "(almost no shared profile keywords detected)"
    return f"""What the three long-form application questions are about (headers only — not the answers):
(1) {q20}
(2) {q21}
(3) {q23}

Retrieval context (wording only — do not repeat as numbers in your answer): {_profile_alignment_words(row)}; {_retrieval_place_words(row)}.

Shared profile keywords (mentee vs mentor combined text): {overlap}

Written-application depth signal (optional, qualitative): {_application_depth_words(row)}.""".strip()


def _finalize_explanation_line(text, max_chars=420):
    """Keep one physical line; preserve the model's full sentence unless extremely long (Excel-friendly cap)."""
    line = str(text).strip().split("\n")[0].strip()
    line = " ".join(line.split())
    if not line:
        return line
    if len(line) <= max_chars:
        return line
    cut = line[:max_chars]
    if " " in cut:
        cut = cut[: cut.rfind(" ")].rstrip()
    return cut + "…"


def explain_top_match_openai(row):
    """Single line; academic/pro focus; no scores in prose; no mentee/mentor boilerplate opening."""
    facts = build_match_explanation_facts(row)
    if openai_client is None:
        return ""
    user_prompt = (
        "Write the single-line summary requested in the system message.\n\n"
        f"{facts}"
    )
    last_error = None
    for attempt in range(1, 4):
        try:
            response = openai_client.chat.completions.create(
                model=EXPLAIN_TOP_MATCH_MODEL,
                temperature=0,
                max_tokens=150,
                messages=[
                    {"role": "system", "content": EXPLAIN_TOP_MATCH_SYSTEM},
                    {"role": "user", "content": user_prompt},
                ],
            )
            raw = (response.choices[0].message.content or "").strip()
            return _finalize_explanation_line(raw)
        except Exception as exc:
            last_error = exc
            if attempt < 3:
                time.sleep(1.5 * attempt)
            else:
                return _finalize_explanation_line(f"(explanation failed: {last_error})")


reranked_candidates = pre_rerank_candidates.copy()
if USE_OPENAI_RERANK:
    reranked_candidates["final_rank_score"] = reranked_candidates.apply(
        lambda row: score_pair_openai(
            mentee_text_map.get(row["mentee"], ""),
            mentor_text_map.get(row["mentor"], "")
        ),
        axis=1
    )
else:
    reranked_candidates["final_rank_score"] = reranked_candidates["total_score"]

# Final pool after final-rank scoring (cap per mentee)
final_candidate_pool = (
    reranked_candidates
    .sort_values(["mentee", "final_rank_score", "total_score", "cosine_norm"], ascending=[True, False, False, False])
    .groupby("mentee", group_keys=False)
    .head(mentee_mentor_cap)
    .copy()
)
final_candidate_pool["final_rank"] = (
    final_candidate_pool.groupby("mentee")["final_rank_score"]
    .rank(method="first", ascending=False)
    .astype(int)
)

rank_df = (
    final_candidate_pool[final_candidate_pool["er_rigtigt_match"]][["mentee", "final_rank"]]
    .rename(columns={"final_rank": "rigtig_mentor_rank"})
)

# Mentor-facing export (3 steps on full scored matrix: 50% cosine + 30% quality + 20% wish):
# 1) Each mentor gets top K mentees by score — no limit on how often a mentee appears across mentors.
# 2) Mentees with >K appearances: drop excess rows with lowest total_score (keep best 5 per mentee).
# 3) Mentors short of K rows: walk that mentor's full score list; add next mentee if not duplicate for mentor
#    and mentee has <5 assignments (skip mentees already at 5).
MENTOR_EXPORT_TOP_K = 3
MENTEE_MAX_MENTOR_TOP3_APPEARANCES = 5


def build_mentor_export_trim_then_backfill(full_scored, slots_per_mentor, max_mentee_appearances):
    df = full_scored.sort_values(
        ["mentor", "total_score", "cosine_norm", "similarity_score"],
        ascending=[True, False, False, False],
    ).reset_index(drop=True)
    df = df.drop_duplicates(subset=["mentor", "mentee"], keep="first")
    mentor_order = sorted(df["mentor"].unique())
    mentor_indices = {m: df.index[df["mentor"] == m].tolist() for m in mentor_order}

    picked_set = set()
    for m in mentor_order:
        for i in mentor_indices[m][:slots_per_mentor]:
            picked_set.add(i)
    temp = df.loc[sorted(picked_set)].copy()
    mx = int(temp.groupby("mentee").size().max()) if len(temp) else 0
    print(
        f"Export step 1 — top-{slots_per_mentor}/mentor (mentee uncapped): {len(temp)} rows, "
        f"max mentee appearances = {mx}"
    )

    to_remove = []
    for _, sub in temp.groupby("mentee"):
        if len(sub) <= max_mentee_appearances:
            continue
        drop_idx = (
            sub.sort_values(["total_score", "cosine_norm", "similarity_score"], ascending=[True, True, True])
            .index[: -max_mentee_appearances]
            .tolist()
        )
        to_remove.extend(drop_idx)
    picked_set -= set(to_remove)
    selected = df.loc[sorted(picked_set)].copy()
    print(
        f"Export step 2 — keep best {max_mentee_appearances} rows/mentee by score: "
        f"removed {len(to_remove)}, {len(selected)} rows left"
    )

    for m in mentor_order:
        while (selected["mentor"] == m).sum() < slots_per_mentor:
            mentee_counts = selected.groupby("mentee").size().to_dict()
            mentees_m = set(selected.loc[selected["mentor"] == m, "mentee"].tolist())
            added = False
            for i in mentor_indices[m]:
                if i in selected.index:
                    continue
                u = df.at[i, "mentee"]
                if u in mentees_m:
                    continue
                if mentee_counts.get(u, 0) >= max_mentee_appearances:
                    continue
                selected = pd.concat([selected, df.loc[[i]]])
                added = True
                break
            if not added:
                print(
                    f"Warning: mentor {m!r} has {(selected['mentor'] == m).sum()} row(s) "
                    f"(<{slots_per_mentor}) — no backfill candidate under cap."
                )
                break

    selected = selected.sort_values(
        ["mentor", "total_score", "cosine_norm", "similarity_score"],
        ascending=[True, False, False, False],
    ).reset_index(drop=True)
    over = selected.groupby("mentee").size()
    over = over[over > max_mentee_appearances]
    if len(over):
        print(f"ERROR: mentees still over cap after pipeline: {over.to_dict()}")
    print(
        f"Export step 3 — after backfill: {len(selected)} rows, "
        f"sum total_score = {selected['total_score'].sum():.2f}, "
        f"max mentee count = {int(selected.groupby('mentee').size().max()) if len(selected) else 0}"
    )
    return selected


_export_scored = scored_candidates.copy()
_export_scored["final_rank"] = (
    _export_scored.groupby("mentee")["total_score"]
    .rank(method="first", ascending=False)
    .astype(int)
)
resultater_top3 = build_mentor_export_trim_then_backfill(
    _export_scored,
    slots_per_mentor=MENTOR_EXPORT_TOP_K,
    max_mentee_appearances=MENTEE_MAX_MENTOR_TOP3_APPEARANCES,
)
resultater_top3["mentor_rank"] = (
    resultater_top3.groupby("mentor")["total_score"]
    .rank(method="first", ascending=False)
    .astype(int)
)
resultater_top3["final_rank_score"] = resultater_top3["total_score"]
resultater_top3["is_historical_match"] = resultater_top3["er_rigtigt_match"].map({True: "SAND", False: "FALSK"})
_mcnt = resultater_top3.groupby("mentor").size()
print(
    f"Mentor export table: {len(resultater_top3)} rows — trim-then-backfill pipeline "
    f"(≤{MENTOR_EXPORT_TOP_K}/mentor; ≤{MENTEE_MAX_MENTOR_TOP3_APPEARANCES}/mentee)."
)
print(
    f"  Mentors with fewer than {MENTOR_EXPORT_TOP_K} rows: {int((_mcnt < MENTOR_EXPORT_TOP_K).sum())}"
)
print(f"  Max mentee appearances across export: {resultater_top3.groupby('mentee').size().max()}")

if USE_OPENAI_TOP_MATCH_EXPLANATION:
    if openai_client is None:
        print("USE_OPENAI_TOP_MATCH_EXPLANATION is True but openai_client is missing; match_explanation left empty.")
        resultater_top3["match_explanation"] = ""
    else:
        print("Generating short OpenAI explanations for exported top-table rows only...")
        resultater_top3["match_explanation"] = resultater_top3.apply(
            lambda row: explain_top_match_openai(row),
            axis=1,
        )
else:
    resultater_top3["match_explanation"] = ""

resultater_top3["shared_keywords"] = resultater_top3.apply(
    lambda row: find_shared_terms(
        mentee_text_map.get(row["mentee"], ""),
        mentor_text_map.get(row["mentor"], ""),
        top_n=8,
    ),
    axis=1,
)

export_columns = [
    "mentor",
    "mentor_rank",
    "mentee",
    "final_rank_score",
    "total_score",
    "cosine_norm",
    "quality_score_final",
    "quality_score_openai",
    "video_score",
    "onske_bonus",
    "retrieval_rank",
    "shared_keywords",
    "final_rank",
    "match_explanation",
    "is_historical_match"
]

resultater_top3 = resultater_top3[export_columns].rename(columns={
    "mentor": "Mentor",
    "mentor_rank": "Mentor Rank",
    "mentee": "Mentee",
    "final_rank_score": "Final Rank Score",
    "total_score": "Total Score (Pre-Final Rank)",
    "cosine_norm": "Cosine Retrieval Score",
    "quality_score_final": "Quality Score (OpenAI + 10% video)",
    "quality_score_openai": "OpenAI text quality (0–10)",
    "video_score": "Video Score",
    "onske_bonus": "Preference Bonus",
    "retrieval_rank": "Cosine Retrieval Rank",
    "shared_keywords": "Shared keywords (profile overlap)",
    "final_rank": "Final Rank",
    "match_explanation": "Match explanation",
    "is_historical_match": "Is Historical Match"
})
resultater_top3 = resultater_top3.sort_values(["Mentor", "Mentor Rank"]).reset_index(drop=True)

total_score_stats = {
    "avg_total_score": final_candidate_pool["total_score"].mean(),
    "median_total_score": final_candidate_pool["total_score"].median(),
    "min_total_score": final_candidate_pool["total_score"].min(),
    "max_total_score": final_candidate_pool["total_score"].max(),
    "avg_final_rank_score": final_candidate_pool["final_rank_score"].mean(),
    "top3_rows": len(resultater_top3),
    "mentor_count": resultater_top3["Mentor"].nunique(),
}

if USE_OPENAI_RERANK:
    print("OpenAI final rank scoring complete.")
else:
    print("Final rank LLM rerank skipped (final_rank_score = total_score).")
    print("Note: export column 'Final Rank Score' is then on the total_score scale, not 0–10.")
if USE_OPENAI_TOP_MATCH_EXPLANATION and openai_client is not None:
    print("Match explanation: one line, academic/pro focus; no numeric cosine in prose; no 'The mentee/mentor' openings; ~420 char soft cap with word-boundary trim.")


OpenAI rerank template prepared (disabled).


## 13. Statistics

In [1]:
print("=== STATISTICS ===")

if "mentees" not in globals():
    raise RuntimeError(
        "Statistik kræver data fra tidligere celler. Kør notebooken fra toppen (Run All / Run Above), "
        "så fx `mentees` og `all_match_candidates` findes — ellers stopper den her."
    )

raw_counts = {
    "raw_mentees": len(mentees),
    "cleaned_mentees": len(mentees_renset),
    "removed_duplicates": len(mentees) - len(mentees_renset),
}

similarity_stats = {
    "total_candidates": len(all_match_candidates),
    "unique_mentees": all_match_candidates["mentee"].nunique(),
    "unique_mentors": all_match_candidates["mentor"].nunique(),
    "avg_similarity": all_match_candidates["similarity_score"].mean(),
    "min_similarity": all_match_candidates["similarity_score"].min(),
    "max_similarity": all_match_candidates["similarity_score"].max(),
    "avg_correct_mentor_total_rank": rank_df["rigtig_mentor_rank"].mean(),
}

candidate_pool_stats = {
    "total_scored_pairs": len(scored_candidates),
    "openai_rerank_top_k": openai_rerank_top_k,
    "pre_rerank_pairs": len(pre_rerank_candidates),
    "mentee_mentor_cap": mentee_mentor_cap,
    "final_candidate_pairs": len(final_candidate_pool),
    "avg_cosine_score": final_candidate_pool["cosine_norm"].mean(),
}

def split_into_sentences(text):
    normalized_text = re.sub(r"\s+", " ", str(text)).strip()
    if not normalized_text:
        return []
    sentences = [
        sentence.strip()
        for sentence in re.split(r"(?<=[.!?])\s+|\n+", normalized_text)
        if sentence.strip()
    ]
    return sentences or [normalized_text]

def extract_tokens(text):
    return {
        token
        for token in re.findall(r"\b[a-z_]{3,}\b", str(text).lower())
        if token not in stop_words
    }

def find_best_sentence_pair(mentee_text, mentor_text):
    mentee_sentences = split_into_sentences(mentee_text)
    mentor_sentences = split_into_sentences(mentor_text)

    if not mentee_sentences or not mentor_sentences:
        return pd.Series({
            "mentee_sentence": "",
            "mentor_sentence": "",
            "shared_keywords": ""
        })

    best_pair = None
    best_score = (-1, -1, -1)

    for mentee_sentence in mentee_sentences:
        mentee_tokens = extract_tokens(mentee_sentence)
        for mentor_sentence in mentor_sentences:
            mentor_tokens = extract_tokens(mentor_sentence)
            shared_tokens = sorted(
                mentee_tokens & mentor_tokens,
                key=lambda token: (token in mine_vigtige_ord, token),
                reverse=True
            )
            overlap_score = len(shared_tokens)
            coverage_score = len(mentee_tokens | mentor_tokens) * -1
            length_score = -(abs(len(mentee_sentence) - 160) + abs(len(mentor_sentence) - 160))
            candidate_score = (overlap_score, coverage_score, length_score)

            if candidate_score > best_score:
                best_score = candidate_score
                best_pair = {
                    "mentee_sentence": mentee_sentence,
                    "mentor_sentence": mentor_sentence,
                    "shared_keywords": ", ".join(shared_tokens[:5])
                }

    if best_pair is None:
        best_pair = {
            "mentee_sentence": mentee_sentences[0],
            "mentor_sentence": mentor_sentences[0],
            "shared_keywords": ""
        }

    best_pair["mentee_sentence"] = re.sub(r"\s+", " ", best_pair["mentee_sentence"]).strip()
    best_pair["mentor_sentence"] = re.sub(r"\s+", " ", best_pair["mentor_sentence"]).strip()
    return pd.Series(best_pair)

top_examples = (
    final_candidate_pool
    .sort_values(["total_score", "cosine_norm"], ascending=[False, False])
    .head(5)
    .copy()
)
top_examples[["mentee_sentence", "mentor_sentence", "shared_keywords"]] = top_examples.apply(
    lambda row: find_best_sentence_pair(
        mentee_text_map.get(row["mentee"], ""),
        mentor_text_map.get(row["mentor"], "")
    ),
    axis=1
)

quality_q1 = mentees_renset["quality_score_final"].quantile(0.25)
quality_q3 = mentees_renset["quality_score_final"].quantile(0.75)
quality_lower_count = int((mentees_renset["quality_score_final"] <= quality_q1).sum())
quality_upper_count = int((mentees_renset["quality_score_final"] >= quality_q3).sum())
quality_middle_count = int(((mentees_renset["quality_score_final"] > quality_q1) & (mentees_renset["quality_score_final"] < quality_q3)).sum())

mentor_onske_popularity = {}
for mentor in mentor_names:
    onske_1_count = (mentees_renset["onske_1"] == mentor).sum()
    onske_2_count = (mentees_renset["onske_2"] == mentor).sum()
    onske_3_count = (mentees_renset["onske_3"] == mentor).sum()
    total = onske_1_count + onske_2_count + onske_3_count
    if total > 0:
        mentor_onske_popularity[mentor] = {
            "onske_1": onske_1_count,
            "onske_2": onske_2_count,
            "onske_3": onske_3_count,
            "total": total
        }

sorted_mentors = sorted(mentor_onske_popularity.items(), key=lambda x: x[1]["total"], reverse=True)
mentor_popularity_df = pd.DataFrame([
    {"mentor": mentor_name, **mentor_onske_popularity[mentor_name]}
    for mentor_name, _ in sorted_mentors[:15]
])

def is_blank_choice(series):
    return series.isna() | series.astype(str).str.strip().eq("")

wish_status_df = mentees_renset[["navn", "onske_1", "onske_2", "onske_3"]].copy()
wish_status_df["has_no_wish_list"] = (
    is_blank_choice(wish_status_df["onske_1"]) &
    is_blank_choice(wish_status_df["onske_2"]) &
    is_blank_choice(wish_status_df["onske_3"])
)
wish_status_df = wish_status_df.rename(columns={"navn": "Mentee"})

preference_rank_source = resultater_top3.merge(
    wish_status_df[["Mentee", "has_no_wish_list"]],
    on="Mentee",
    how="left"
)

def preference_bucket(row):
    if bool(row["has_no_wish_list"]):
        return "No wish list"
    bonus = int(row["Preference Bonus"])
    if bonus == 0:
        return "0 (had wishes)"
    return str(bonus)

preference_rank_source["preference_bucket"] = preference_rank_source.apply(preference_bucket, axis=1)
preference_distribution_df = pd.crosstab(
    preference_rank_source["preference_bucket"],
    preference_rank_source["Mentor Rank"].astype(int)
).reindex(index=["0 (had wishes)", "4", "7", "10", "No wish list"], columns=[1, 2, 3], fill_value=0)

ranked_mentees = set(resultater_top3["Mentee"].dropna().unique())
all_mentees = set(mentees_renset["navn"].dropna().unique())
not_ranked_mentees = all_mentees - ranked_mentees

wish_pairs = (
    wish_status_df.melt(
        id_vars=["Mentee", "has_no_wish_list"],
        value_vars=["onske_1", "onske_2", "onske_3"],
        value_name="Mentor"
    )
    .drop(columns=["variable"])
    .assign(Mentor=lambda df: df["Mentor"].astype(str).str.strip())
    .loc[lambda df: (~df["has_no_wish_list"]) & df["Mentor"].ne("") & df["Mentor"].ne("nan")]
    .drop_duplicates(["Mentee", "Mentor"])
)

wish_top3_hits = wish_pairs.merge(
    resultater_top3[["Mentee", "Mentor", "Mentor Rank"]],
    on=["Mentee", "Mentor"],
    how="left"
)
mentees_with_wishes = set(wish_pairs["Mentee"].unique())
mentees_with_wish_top3_hit = set(wish_top3_hits.loc[wish_top3_hits["Mentor Rank"].notna(), "Mentee"].unique())
mentees_with_wishes_not_in_desired_top3 = mentees_with_wishes - mentees_with_wish_top3_hit

ranking_coverage_stats = {
    "unique_mentees_in_ranking": len(ranked_mentees),
    "unique_mentees_not_ranked": len(not_ranked_mentees),
    "mentees_with_wishes": len(mentees_with_wishes),
    "mentees_with_wishes_not_in_desired_top3": len(mentees_with_wishes_not_in_desired_top3),
}

print("\nPopulation:")
print(f"  Raw mentee rows in Excel: {raw_counts['raw_mentees']}")
print(f"  Unique mentees after dedup/cleaning: {raw_counts['cleaned_mentees']}")
print(f"  Removed duplicate mentee rows: {raw_counts['removed_duplicates']}")

print("\nScoring and ranking:")
print(f"  Total candidate pairs scored: {candidate_pool_stats['total_scored_pairs']}")
print(f"  Top-k per mentee sent to OpenAI final rank: {candidate_pool_stats['openai_rerank_top_k']}")
print(f"  Candidate pairs before final rank (top-k pool): {candidate_pool_stats['pre_rerank_pairs']}")
print(f"  Max mentors kept per mentee after final rank: {candidate_pool_stats['mentee_mentor_cap']}")
print(f"  Candidate pairs after mentee cap: {candidate_pool_stats['final_candidate_pairs']}")
print(f"  Avg cosine score in final pool: {candidate_pool_stats['avg_cosine_score']:.2f}/10")

print("\nCosine retrieval:")
print(f"  Total candidates before filtering: {similarity_stats['total_candidates']}")
print(f"  Unique mentees: {similarity_stats['unique_mentees']}")
print(f"  Unique mentors: {similarity_stats['unique_mentors']}")
print(f"  Avg similarity: {similarity_stats['avg_similarity']:.4f}")
print(f"  Min similarity: {similarity_stats['min_similarity']:.4f}")
print(f"  Max similarity: {similarity_stats['max_similarity']:.4f}")
print(f"  Avg rank of correct mentor by total score: {similarity_stats['avg_correct_mentor_total_rank']:.1f}")
print("  Top 5 ranked examples with one sentence from each side:")
print(top_examples[[
    "mentee",
    "mentor",
    "cosine_norm",
    "total_score",
    "shared_keywords",
    "mentee_sentence",
    "mentor_sentence"
]].to_string(index=False))

print("\nRanking coverage:")
print(f"  Unique mentees in final ranking top 3: {ranking_coverage_stats['unique_mentees_in_ranking']}")
print(f"  Unique cleaned mentees not appearing in final top 3: {ranking_coverage_stats['unique_mentees_not_ranked']}")
print(f"  Coverage check (ranked + not ranked): {ranking_coverage_stats['unique_mentees_in_ranking'] + ranking_coverage_stats['unique_mentees_not_ranked']}")
print(f"  Mentees with at least one wished mentor: {ranking_coverage_stats['mentees_with_wishes']}")
print(f"  Mentees with wishes but not in top 3 for any wished mentor: {ranking_coverage_stats['mentees_with_wishes_not_in_desired_top3']}")

print("\nQuality score:")
print(f"  Avg combined quality score: {mentees_renset['quality_score_final'].mean():.2f}/10")
print(f"  Avg OpenAI text-only quality: {mentees_renset['quality_score_openai'].mean():.2f}/10")
print(f"  Median: {mentees_renset['quality_score_final'].median():.2f}/10")
print(f"  Lower quartile (Q1): {quality_q1:.2f}/10")
print(f"  Upper quartile (Q3): {quality_q3:.2f}/10")
print(f"  Lower quartile count: {quality_lower_count} mentees ({100 * quality_lower_count / len(mentees_renset):.1f}%)")
print(f"  Middle 50% count: {quality_middle_count} mentees ({100 * quality_middle_count / len(mentees_renset):.1f}%)")
print(f"  Upper quartile count: {quality_upper_count} mentees ({100 * quality_upper_count / len(mentees_renset):.1f}%)")
print(f"  Min: {mentees_renset['quality_score_final'].min():.2f}/10")
print(f"  Max: {mentees_renset['quality_score_final'].max():.2f}/10")
print("  Distribution:")
for threshold in [2, 4, 6, 8]:
    count = (mentees_renset["quality_score_final"] >= threshold).sum()
    pct = 100 * count / len(mentees_renset)
    print(f"    {threshold}+ : {count} mentees ({pct:.1f}%)")

print("\nMentor onske popularity:")
print(mentor_popularity_df.to_string(index=False))

print("\nTotal score:")
print(f"  Avg total score (pre-final rank): {total_score_stats['avg_total_score']:.2f}/10")
print(
    f"  Avg final rank score: {total_score_stats['avg_final_rank_score']:.2f}"
    + (" /10" if USE_OPENAI_RERANK else " (same scale as total score; LLM rerank off)")
)
print(f"  Median total score: {total_score_stats['median_total_score']:.2f}/10")
print(f"  Min total score: {total_score_stats['min_total_score']:.2f}/10")
print(f"  Max total score: {total_score_stats['max_total_score']:.2f}/10")
print(f"  Final output rows: {total_score_stats['top3_rows']}")
print(f"  Mentors covered: {total_score_stats['mentor_count']}")
print("  Distribution:")
for threshold in [2, 4, 6, 8]:
    count = (final_candidate_pool["total_score"] >= threshold).sum()
    pct = 100 * count / len(final_candidate_pool)
    print(f"    {threshold}+ : {count} candidate matches ({pct:.1f}%)")
print("  Preference bonus by mentor rank:")
print(preference_distribution_df.to_string())

print("\nScore components:")
print("  Pre-final-rank total score = 50% cosine + 30% quality + 20% preference bonus")
print("    (quality = 90% OpenAI text score + 10% video_score, same 90/10 split as old ML vs video)")
print("  OpenAI final rank is applied after total score on top-k per mentee")
print(f"    - k (top-k): {openai_rerank_top_k}")
print(f"    - Final cap after final rank: {mentee_mentor_cap} per mentee")
print("  Preference bonus (graduated):")
print("    - 1st choice: 10 points")
print("    - 2nd choice: 7 points")
print("    - 3rd choice: 4 points")
print("    - No match: 0 points")

=== STATISTICS ===


NameError: name 'mentees' is not defined

## 14. Export Per Mentor

In [15]:
output_path = "/Users/maltejuuljohansen/Desktop/Conflux/Matching_Results.xlsx"
with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    resultater_top3.to_excel(writer, sheet_name='Top 3 Mentees Per Mentor', index=False)

print(f"Exported to: {output_path}")

Exported to: /Users/maltejuuljohansen/Desktop/Conflux/Matching_Results.xlsx
